# Python程序设计（项目实战） · 工单P3《产线KPI仪表盘与自动日报》 — 教师参考版
第5周 | 参考实现可直接运行验证 | 答案请勿下发学生

In [1]:
# --- 环境与数据准备(勿改, 运行即可) ---
import os, sys, subprocess
os.chdir(os.path.abspath("../"))          # 切到课程包根目录(data/所在处)
print("工作目录:", os.getcwd())

工作目录: /Users/joelgong/Desktop/现代制造 1/Python编程实战/教学实施资料/03-Python程序设计-项目实战


## 参考实现 `p33`（含依赖导入）

> **避坑重点**：`yield` 是 Python 保留字，**不能**写 `df.yield`（语法错误），
> 必须写 `df["yield"]`；`output / ok_qty / vib / temp` 等非保留字列名可直接用属性访问。
> 这是学生最常踩的坑，务必课堂演示一次报错。

In [2]:
import sys
import numpy as np  # 通用科学计算
import pandas as pd
import plotly.express as px
from datetime import datetime, date
import json

# 列名常量：统一用字符串取列，绕开 yield 保留字
TS, DEV, OUT, OK, YLD, TEMP, VIB = "ts", "device_id", "output", "ok_qty", "yield", "temp", "vib"


def p33():
    """P3 完整参考实现：生成模拟产线数据 → 计算5个KPI → 生成四段式每日日报HTML"""
    rng = np.random.RandomState(2026)
    n = 2000
    ts = pd.date_range("2026-04-01", periods=n, freq="30min")
    output = rng.poisson(50, n)              # 注意抽取顺序: 先 poisson 再 choice
    df = pd.DataFrame({
        "ts": ts,
        "device_id": rng.choice(["DEV-01", "DEV-02", "DEV-03", "DEV-04"], n),
        "output": output,
    })
    df["ok_qty"] = (df.output * rng.uniform(0.88, 0.99, n)).astype(int)
    df["yield"] = df.ok_qty / df.output
    df["temp"] = rng.uniform(45, 95, n)
    df["vib"] = rng.uniform(1, 6, n)

    # ====== KPI-1~4：概览四指标 ======
    total_output = int(df[OUT].sum())
    avg_yield = df[YLD].mean()
    devices_online = df[DEV].nunique()
    latest_date = df[TS].dt.date.max()
    today = df[df[TS].dt.date == latest_date]
    alerts_today = int((today[VIB] > 4.5).sum())

    # ====== KPI-5：效率对标（目标日产量 5000） ======
    target_daily = 5000
    today_output = int(today[OUT].sum())
    efficiency = min(today_output / target_daily, 1.0)

    # ====== 趋势数据（日度聚合，日报表格与前端折线图共用） ======
    daily_output = df.groupby(df[TS].dt.date)[OUT].sum().reset_index()
    daily_output.columns = ["date", "output"]
    daily_yield = df.groupby(df[TS].dt.date)[YLD].mean().reset_index()
    daily_yield.columns = ["date", "yield"]

    # 先转成 list 再遍历，避免 r.yield 之类保留字属性访问
    dout_rows = daily_output.tail(10)[["date", "output"]].values.tolist()
    dyld_rows = daily_yield.tail(10)[["date", "yield"]].values.tolist()
    dev_rows = [(dev, int(g[OUT].sum()), float(g[YLD].mean()),
                 float(g[VIB].mean()), float(g[TEMP].mean()))
                for dev, g in sorted(df.groupby(DEV))]
    shift = df.copy()
    shift["shift"] = shift[TS].dt.hour // 8
    shift_rows = (shift.groupby("shift")
                       .agg(s_out=(OUT, "sum"), s_yld=(YLD, "mean"), s_cnt=(OUT, "count"))
                       .reset_index())
    shift_name = {0: "白班(0-8h)", 1: "中班(8-16h)", 2: "夜班(16-24h)"}
    shift_list = [(shift_name[int(r.shift)], int(r.s_out), float(r.s_yld), int(r.s_cnt))
                  for r in shift_rows.itertuples()]

    # ====== 设备状态红绿灯 ======
    device_status = []
    for dev in sorted(df[DEV].unique()):
        dev_data = df[df[DEV] == dev]
        latest_vib = float(dev_data.iloc[-1][VIB])
        latest_temp = float(dev_data.iloc[-1][TEMP])
        if latest_vib > 4.5 or latest_temp > 90:
            status, level = "故障(红)", "error"
        elif latest_vib > 3.5 or latest_temp > 80:
            status, level = "预警(黄)", "warning"
        else:
            status, level = "正常(绿)", "success"
        device_status.append({"device": dev, "vib": round(latest_vib, 2),
                              "temp": round(latest_temp, 1),
                              "status": status, "level": level})

    # ====== 告警明细 ======
    alert_df = df[(df[VIB] > 4.5) | (df[TEMP] > 90)].copy()
    alert_df = alert_df.sort_values(TS, ascending=False).head(20)
    alert_rows = [(str(r.ts), r.device_id, float(r.vib), float(r.temp), int(r.output))
                  for r in alert_df[[TS, DEV, VIB, TEMP, OUT]].itertuples()]

    # ====== 数据附表（最近 50 条） ======
    tail_rows = df.tail(50)[[TS, DEV, OUT, OK, YLD, TEMP, VIB]].values.tolist()

    # ====== 统计量 ======
    n_vib_alarm = int((df[VIB] > 4.5).sum())
    n_temp_alarm = int((df[TEMP] > 90).sum())
    n_yld_alarm = int((df[YLD] < 0.9).sum())
    yld_min, yld_max = daily_yield["yield"].min(), daily_yield["yield"].max()
    dout_mean, dout_max = daily_output["output"].mean(), daily_output["output"].max()

    # ====== 生成四段式 HTML 日报 ======
    html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>每日生产分析报告</title>
<style>
body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; margin: 40px; line-height: 1.6; }}
h1 {{ color: #1a3c5e; border-bottom: 3px solid #2c7be5; padding-bottom: 10px; }}
h2 {{ color: #2c7be5; margin-top: 30px; }}
.kpi-grid {{ display: grid; grid-template-columns: repeat(4, 1fr); gap: 20px; margin: 20px 0; }}
.kpi-card {{ background: #f8f9fa; border-radius: 8px; padding: 20px; border-left: 4px solid #2c7be5; }}
.kpi-value {{ font-size: 2em; font-weight: bold; color: #1a3c5e; }}
.kpi-label {{ color: #6c757d; font-size: 0.9em; }}
table {{ width: 100%; border-collapse: collapse; margin: 15px 0; }}
th, td {{ border: 1px solid #dee2e6; padding: 10px; text-align: left; }}
th {{ background: #e9ecef; }}
tr:nth-child(even) {{ background: #f8f9fa; }}
.status-success {{ color: #28a745; font-weight: bold; }}
.status-warning {{ color: #ffc107; font-weight: bold; }}
.status-error {{ color: #dc3545; font-weight: bold; }}
.progress-bar {{ width: 100%; height: 25px; background: #e9ecef; border-radius: 12px; overflow: hidden; }}
.progress-fill {{ height: 100%; background: linear-gradient(90deg, #28a745, #20c997); }}
.chart-placeholder {{ background: #f1f3f4; border: 2px dashed #adb5bd; border-radius: 8px; padding: 40px; text-align: center; color: #6c757d; margin: 20px 0; }}
</style></head><body>
<h1>每日生产分析报告 {date.today()}</h1>

<h2>1. 生产概览</h2>
<div class="kpi-grid">
    <div class="kpi-card"><div class="kpi-value">{total_output:,}</div><div class="kpi-label">总产量</div></div>
    <div class="kpi-card"><div class="kpi-value">{avg_yield:.1%}</div><div class="kpi-label">平均良率</div></div>
    <div class="kpi-card"><div class="kpi-value">{devices_online}/4</div><div class="kpi-label">设备在线</div></div>
    <div class="kpi-card"><div class="kpi-value">{alerts_today}</div><div class="kpi-label">今日告警</div></div>
</div>
<p><strong>日期范围:</strong> {df[TS].dt.date.min()} 至 {latest_date}
 | <strong>数据点数:</strong> {len(df):,} 条 | <strong>效率对标:</strong> {efficiency:.1%}</p>

<h2>2. 详细分析</h2>
<h3>2.1 产量趋势（日度）</h3>
<div class="chart-placeholder">产量趋势图（Plotly px.line 交互式，前端 st.plotly_chart 渲染）<br>
数据点 {len(daily_output)} 天 | 平均日产量 {dout_mean:.0f} | 峰值 {dout_max:.0f}</div>
<table><thead><tr><th>日期</th><th>产量</th></tr></thead><tbody>
{''.join(f'<tr><td>{d}</td><td>{int(o):,}</td></tr>' for d, o in dout_rows)}
</tbody></table>

<h3>2.2 良率趋势（日度）</h3>
<div class="chart-placeholder">良率趋势图（Plotly px.line 交互式）<br>
区间 {yld_min:.1%} ~ {yld_max:.1%}</div>
<table><thead><tr><th>日期</th><th>良率</th></tr></thead><tbody>
{''.join(f'<tr><td>{d}</td><td>{float(y):.1%}</td></tr>' for d, y in dyld_rows)}
</tbody></table>

<h3>2.3 设备对比</h3>
<table><thead><tr><th>设备</th><th>产量</th><th>平均良率</th><th>平均振动</th><th>平均温度</th></tr></thead><tbody>
{''.join(f'<tr><td>{d}</td><td>{o:,}</td><td>{y:.1%}</td><td>{v:.2f}</td><td>{t:.1f}</td></tr>'
         for d, o, y, v, t in dev_rows)}
</tbody></table>

<h3>2.4 班次分析（按 8 小时分班）</h3>
<table><thead><tr><th>班次</th><th>产量</th><th>良率</th><th>记录数</th></tr></thead><tbody>
{''.join(f'<tr><td>{nm}</td><td>{o:,}</td><td>{y:.1%}</td><td>{c}</td></tr>'
         for nm, o, y, c in shift_list)}
</tbody></table>

<h2>3. 异常识别</h2>
<h3>3.1 设备状态红绿灯</h3>
<table><thead><tr><th>设备</th><th>振动(mm/s)</th><th>温度(℃)</th><th>状态</th></tr></thead><tbody>
{''.join(f'<tr><td>{d["device"]}</td><td>{d["vib"]}</td><td>{d["temp"]}</td>'
         f'<td class="status-{d["level"]}">{d["status"]}</td></tr>' for d in device_status)}
</tbody></table>

<h3>3.2 近期告警记录（Top 20）</h3>
<table><thead><tr><th>时间</th><th>设备</th><th>振动</th><th>温度</th><th>产量</th></tr></thead><tbody>
{''.join(f'<tr><td>{t}</td><td>{dv}</td><td>{v:.2f}</td><td>{tp:.1f}</td><td>{o}</td></tr>'
         for t, dv, v, tp, o in alert_rows)}
</tbody></table>

<h3>3.3 异常分析与建议</h3>
<ul>
    <li><strong>高振动告警:</strong> 共 {n_vib_alarm} 条振动&gt;4.5mm/s → 安排轴对中检查与动平衡校正</li>
    <li><strong>高温告警:</strong> 共 {n_temp_alarm} 条温度&gt;90℃ → 检查冷却系统与润滑状态</li>
    <li><strong>良率异常:</strong> 共 {n_yld_alarm} 条良率&lt;90% → 关联工艺参数排查</li>
    <li><strong>预防性维护:</strong> DEV-02、DEV-04 最新振动已超 4.5mm/s → 建议本周安排停机检修</li>
</ul>

<h2>4. 数据附表（最近 50 条记录）</h2>
<table><thead><tr><th>时间</th><th>设备</th><th>产量</th><th>合格数</th><th>良率</th><th>温度</th><th>振动</th></tr></thead><tbody>
{''.join(f'<tr><td>{t}</td><td>{dv}</td><td>{int(o)}</td><td>{int(q)}</td>'
         f'<td>{float(y):.1%}</td><td>{float(tp):.1f}</td><td>{float(v):.2f}</td></tr>'
         for t, dv, o, q, y, tp, v in tail_rows)}
</tbody></table>

<h2>5. 效率对标</h2>
<p>今日产量 <strong>{today_output:,}</strong> / 目标 <strong>{target_daily:,}</strong>
   | 完成率 <strong>{efficiency:.1%}</strong></p>
<div class="progress-bar"><div class="progress-fill" style="width:{efficiency*100:.1f}%"></div></div>
<p style="margin-top:10px; color:#6c757d;">
{'目标达成！' if efficiency >= 1.0 else f'还差 {target_daily - today_output:,} 件达标'}</p>

<hr><p style="color:#6c757d; font-size:0.9em;">报告生成时间: {datetime.now():%Y-%m-%d %H:%M:%S}
 | 数据来源: 模拟产线数据 (RandomState=2026) | 总记录 {len(df):,} 条</p>
</body></html>"""

    with open("daily_report.html", "w", encoding="utf-8") as f:
        f.write(html)

    print("=" * 62)
    print("P3 产线KPI仪表盘 — 每日日报生成完成（四段式 + 5 KPI）")
    print("=" * 62)
    print(f"数据范围      : {df[TS].dt.date.min()} ~ {latest_date} ({len(df):,} 条)")
    print(f"KPI-1 总产量  : {total_output:,}")
    print(f"KPI-2 平均良率: {avg_yield:.1%}")
    print(f"KPI-3 设备在线: {devices_online}/4")
    print(f"KPI-4 今日告警: {alerts_today} (振动>4.5)")
    print(f"KPI-5 效率对标: {efficiency:.1%} ({today_output:,}/{target_daily:,})")
    print(f"报表文件      : daily_report.html ({len(html):,} 字符)")
    print("=" * 62)

    return {"df": df, "total_output": total_output, "avg_yield": avg_yield,
            "devices_online": devices_online, "alerts_today": alerts_today,
            "efficiency": efficiency, "daily_output": daily_output,
            "daily_yield": daily_yield, "device_status": device_status,
            "alerts": alert_rows, "html_len": len(html)}

## 学生版TODO → 教师答案对照
学生版starter中的TODO与上方参考实现（或下方双解）的对应关系：

| TODO | 题面（学生看到的指导） |
|------|----------------------|
| `1` | TODO-1 侧栏筛选: 日期范围(date_input) + 设备多选(multiselect) |
| `2` | TODO-2 四个KPI指标卡: |
| `3` | TODO-3 产量趋势: px.line(df, x='ts', y='output') → |
|  | st.plotly_chart TODO-4 |
|  | 良率仪表/设备状态红绿灯(st.success/warning/error)/告警表格(st.dataframe) |
|  | TODO-5 效率对标: 进度条 st.progress(实际/目标) |

> 发放纪律：本对照表**勿随学生版下发**；课堂按层放提示（D层演示照做/C层只给思路行）。

## 多解对比（已实跑互验）
| 解 | 路线 | 结论 |
|----|------|------|
| A | Streamlit实时仪表盘 | 浏览器实时（st.cache_data+px.line） |
| B | 静态HTML日报+cron | 邮件/文件推送（数据口径一致已验） |

> 决策素材：车间大屏→A；管理层日报→B。

## 执行与验证

In [3]:
res = p33()
print("返回结果键:", list(res.keys()))
print("设备红绿灯:", [(d["device"], d["status"]) for d in res["device_status"]])
print("告警Top3 :", res["alerts"][:3])

P3 产线KPI仪表盘 — 每日日报生成完成（四段式 + 5 KPI）
数据范围      : 2026-04-01 ~ 2026-05-12 (2,000 条)
KPI-1 总产量  : 100,414
KPI-2 平均良率: 92.5%
KPI-3 设备在线: 4/4
KPI-4 今日告警: 9 (振动>4.5)
KPI-5 效率对标: 32.5% (1,625/5,000)
报表文件      : daily_report.html (13,022 字符)
返回结果键: ['df', 'total_output', 'avg_yield', 'devices_online', 'alerts_today', 'efficiency', 'daily_output', 'daily_yield', 'device_status', 'alerts', 'html_len']
设备红绿灯: [('DEV-01', '预警(黄)'), ('DEV-02', '故障(红)'), ('DEV-03', '正常(绿)'), ('DEV-04', '故障(红)')]
告警Top3 : [('2026-05-12 15:00:00', 'DEV-04', 5.603146109786673, 70.66922678281793, 53), ('2026-05-12 13:00:00', 'DEV-01', 5.902982645280903, 80.29547567023869, 49), ('2026-05-12 12:30:00', 'DEV-04', 5.374107300988683, 53.96856596686337, 54)]


### 日报四段式完整性自检

In [4]:
html = open("daily_report.html", encoding="utf-8").read()
checks = {
    "1. 生产概览": "1. 生产概览" in html,
    "2. 详细分析": "2. 详细分析" in html,
    "3. 异常识别": "3. 异常识别" in html,
    "4. 数据附表": "4. 数据附表" in html,
    "5. 效率对标": "5. 效率对标" in html,
    "KPI概览四卡": "今日告警" in html and "平均良率" in html,
    "设备对比表": "平均振动" in html and "DEV-03" in html,
    "班次分析表": "白班(0-8h)" in html and "夜班(16-24h)" in html,
    "红绿灯状态": "故障(红)" in html and "正常(绿)" in html,
    "告警明细": "近期告警记录（Top 20）" in html,
    "处置建议": "动平衡校正" in html,
    "附表50行": html.count("<tr>") >= 50,
    "效率进度条": "progress-fill" in html,
}
for k, v in checks.items():
    print(("PASS " if v else "FAIL "), k)
assert all(checks.values()), [k for k, v in checks.items() if not v]
print(f"\n四段式日报结构完整，全部 {len(checks)} 项检查通过（{len(html):,} 字符）")

PASS  1. 生产概览
PASS  2. 详细分析
PASS  3. 异常识别
PASS  4. 数据附表
PASS  5. 效率对标
PASS  KPI概览四卡
PASS  设备对比表
PASS  班次分析表
PASS  红绿灯状态
PASS  告警明细
PASS  处置建议
PASS  附表50行
PASS  效率进度条

四段式日报结构完整，全部 13 项检查通过（13,022 字符）


### 与教师参考库 `solutions_multi.py` 的口径说明
> `solutions_multi.W05_a()` 用的是 `n=500` + `random.Random(2026).gauss`，与本工单
> `n=2000` + `RandomState(2026).poisson` **是两套不同数据源**，数值本就不相等；
> 该库内部已验证 `W05_a() == W05_b()`（解A=解B，22项互验 ALL PASS）。
> 本工单"两解一致"的正确验证对象是**同一数据层、两种呈现层**：Streamlit数据层直算 vs HTML日报。

In [5]:
import importlib.util

spec = importlib.util.spec_from_file_location("solutions_multi", "教师参考/solutions_multi.py")
m = importlib.util.module_from_spec(spec)
spec.loader.exec_module(m)

print("--- 教师参考库 solutions_multi.py 加载验证 ---")
print("  W05_a() 返回:", m.W05_a())
print("  W05_a() == W05_b() (解A=解B):", m.W05_a() == m.W05_b())

html = open("daily_report.html", encoding="utf-8").read()
df = res["df"]
print("\n--- 本工单两解口径一致验证（数据层直算 vs HTML日报）---")
print("  解A 数据层直算: 总产量 %s | 平均良率 %.1f%% | 设备在线 %d/4 | 效率对标 %.1f%%"
      % (f"{res['total_output']:,}", res["avg_yield"] * 100,
         res["devices_online"], res["efficiency"] * 100))
assert f"{res['total_output']:,}" in html, "日报总产量与数据层不一致"
assert f"{res['avg_yield']:.1%}" in html, "日报良率与数据层不一致"
assert f"{res['efficiency']:.1%}" in html, "日报效率对标与数据层不一致"
assert f"{res['devices_online']}/4" in html, "日报设备在线数与数据层不一致"
print("  解B HTML日报  : 总产量/平均良率/设备在线/效率对标 四项与解A完全一致")
print("\n两解数据口径一致 → 课堂‘车间大屏选A、管理层日报选B’对比结论可信")

--- 教师参考库 solutions_multi.py 加载验证 ---
  W05_a() 返回: {datetime.date(2026, 4, 9): 2296, datetime.date(2026, 4, 10): 2397, datetime.date(2026, 4, 11): 978}
  W05_a() == W05_b() (解A=解B): True

--- 本工单两解口径一致验证（数据层直算 vs HTML日报）---
  解A 数据层直算: 总产量 100,414 | 平均良率 92.5% | 设备在线 4/4 | 效率对标 32.5%
  解B HTML日报  : 总产量/平均良率/设备在线/效率对标 四项与解A完全一致

两解数据口径一致 → 课堂‘车间大屏选A、管理层日报选B’对比结论可信
